# Bayesian A/B Testing Methods — Practice Skeleton

**Goal:** Investigate Bayesian approaches to A/B testing, implement the Beta-Binomial model, compute decision metrics, and explore how priors and sample size affect conclusions.

Work through each step.  Replace every `None` / `TODO` with working code.  The companion **Solution** notebook shows complete answers, alternate implementations, extra practice and a simulation playground.

---


## Flowchart: Bayesian A/B Testing Decision Process

This flowchart shows the recommended Bayesian workflow.  Unlike classical fixed-horizon tests, Bayesian methods allow continuous monitoring and produce directly interpretable probability statements that are easy to communicate to different audiences.

```mermaid
flowchart TD
    A[Start: Business Goal<br/>e.g. Raise conversion rate] --> B[Choose Prior for each variant<br/>Beta(α, β) – weakly informative<br/>or informed by historical data]
    B --> C[Collect Data Sequentially<br/>or in batches<br/>successes / trials for A & B]
    C --> D[Update Posteriors<br/>Beta(α + successes, β + failures)]
    D --> E[Compute Decision Metrics<br/>• P(B > A)<br/>• Expected Loss of choosing wrong<br/>• Credible intervals / ROPE]
    E --> F{Decision Criterion Met?<br/>e.g. P(B>A) > 0.95<br/>or expected loss < threshold}
    F -->|Yes| G[Stop & Decide<br/>Implement winner or keep control]
    F -->|No – continue| C
    G --> H[Write Report<br/>Tailor to Audience:<br/>• Executives: P(better) + expected lift<br/>• Technical: full posteriors, priors, diagnostics<br/>• Mixed: layered + visualisations]
    H --> I[End: Update Organisation Prior<br/>for next experiment]
    style F fill:#fff3cd,stroke:#856404
    style H fill:#e6f3ff,stroke:#0066cc
```

**Key advantage over frequentist:** You can peek continuously.  The posterior already incorporates all information; there is no Type-I error inflation from sequential looks.  Decision thresholds (e.g. P(B>A) > 95 %) are chosen for business risk tolerance, not arbitrary α.


## Audience Considerations (from the provided PDFs)

Bayesian results are especially audience-friendly because they speak in probabilities:

1. **Data Literacy**  
   - High: show full posterior densities, HDI, prior sensitivity.  
   - Low: “There is a 94 % probability that the new design converts better; the most likely lift is +2.1 pp.”

2. **Subject Knowledge**  
   - Experts: discuss prior choice, expected loss, and decision thresholds.  
   - Novices: avoid “Beta(1,1)” jargon; translate everything into “chance the new version is better”.

3. **Time Span**  
   - C-level (30 s): one number – P(better) – plus a clear recommendation.  
   - Technical peer: full derivation, code, and sensitivity analysis.

Use the layered report structure (Introduction → Body by question → Conclusion → Appendix) so each audience can stop at the depth they need.


## Theory: Why Bayesian for A/B Testing?

### Frequentist limitations that Bayesian addresses
- Fixed sample size or complex sequential designs (group sequential, always-valid p-values).
- p-value is **not** the probability that the null is true.
- “Statistically significant” does not tell you the magnitude or the risk of acting.

### Bayesian advantages
1. **Direct probability statements**: “P(treatment > control | data) = 0.93”.
2. **Continuous monitoring**: update the posterior after every observation; stop when a decision threshold is crossed.
3. **Incorporates prior knowledge**: historical conversion rates become the prior; new data update it.
4. **Decision-theoretic**: expected loss quantifies the cost of choosing the wrong variant.
5. **Natural for small samples / rare events**: the prior regularises estimates.

### Core model (conversion rates)
- Likelihood: Binomial (or Bernoulli) for each variant.
- Prior: Beta(α₀, β₀) – conjugate, so posterior is also Beta.
- Posterior mean = (α₀ + successes) / (α₀ + β₀ + trials) – a weighted average of prior mean and observed rate.

Common weakly-informative prior: Beta(1,1) = Uniform(0,1).  
Informed prior: Beta(observed_successes, observed_failures) from a previous period, or a sceptical prior centred near the historical rate.


## 1. Setup & Simulated Experiment Data

We will analyse a hypothetical A/B test for a landing-page change.

### TODO
1. Import `numpy as np`, `scipy.stats.beta`, and `matplotlib.pyplot as plt`.
2. Define the observed data (you may change these later in the simulation):
   - Control: 1200 visitors, 180 conversions
   - Treatment: 1180 visitors, 205 conversions


In [ ]:
# TODO: imports
import numpy as np
from scipy.stats import beta
import matplotlib.pyplot as plt

# TODO: observed data
control_visitors = None
control_conversions = None
treatment_visitors = None
treatment_conversions = None

print("Data loaded (replace Nones to proceed).")


## 2. Choose Priors

We start with a weakly informative prior: Beta(1, 1) (Uniform).

### TODO
1. Set `prior_alpha = 1`, `prior_beta = 1` for both variants.
2. (Optional later) try an informed prior, e.g. Beta(20, 80) reflecting a historical ~20 % rate.


In [ ]:
# TODO: priors
prior_alpha = None
prior_beta  = None
print("Priors set.")


## 3. Update to Posteriors

Posterior for a Beta-Binomial model:

$$
\text{Posterior} = \text{Beta}(\alpha_0 + \text{successes},\; \beta_0 + \text{failures})
$$

### TODO
1. Compute posterior parameters for control and treatment.
2. Compute the posterior means (expected conversion rates).


In [ ]:
# TODO: posterior parameters
post_control_a = None
post_control_b = None
post_treat_a   = None
post_treat_b   = None

# TODO: posterior means
mean_control = None
mean_treat   = None
print("Posterior means:", mean_control, mean_treat)


## 4. Decision Metrics

### Probability that treatment is better
Monte-Carlo estimate: draw many samples from each posterior and count how often treatment > control.

### Expected loss
If we choose treatment when control is actually better, the loss is the difference in rates (and vice-versa).

### TODO
1. Draw 50 000 samples from each posterior.
2. Compute `prob_treat_better = mean(samples_treat > samples_control)`.
3. Compute expected loss of choosing treatment and of choosing control.


In [ ]:
# TODO: Monte-Carlo decision metrics
n_mc = 50000
samples_control = None
samples_treat   = None

prob_treat_better = None
print("P(treatment > control) =", prob_treat_better)

# Expected loss (absolute rate difference)
loss_choose_treat = None   # E[max(control - treat, 0)]
loss_choose_control = None # E[max(treat - control, 0)]
print("Expected loss if we choose treatment:", loss_choose_treat)
print("Expected loss if we choose control  :", loss_choose_control)


## 5. Visualise the Posteriors

### TODO
Plot the two posterior density functions on the same axes.  Add a vertical line at each posterior mean.


In [ ]:
# TODO: plot posteriors
x = np.linspace(0, 0.3, 500)
# density_control = beta.pdf(x, post_control_a, post_control_b)
# density_treat   = beta.pdf(x, post_treat_a, post_treat_b)
# plt.plot(...)
print("Plot the densities (see solution for a complete example).")


## 6. Alternate Code Paths

### Alternate A – analytic approximation for P(B>A)
For large counts the Beta can be approximated by a Normal; you can then use a closed-form expression involving the difference of means and variances.

### Alternate B – use the incomplete beta function / numerical integration instead of Monte-Carlo.

### TODO
Implement at least one alternate and verify it gives a similar probability.


In [ ]:
# TODO: alternate calculation of P(B > A)
pass


## 7. Extra Practice Questions

1. Change the prior to a strongly sceptical Beta(50, 200).  How does P(treat > control) change?
2. Suppose you had only 200 visitors per arm with the same conversion rates.  Recompute the metrics – does the decision change?
3. What decision threshold (on P or on expected loss) would you recommend for a low-stakes UI tweak versus a high-stakes pricing change?
4. Write a 20-second verbal summary suitable for a C-level audience.


In [ ]:
# TODO: answer extra-practice questions
pass


## 8. Simulation Playground – Turn the Knobs

Modify the values below and re-run to explore sensitivity to prior strength, sample size and observed rates.

```python
sim_prior_a = 1
sim_prior_b = 1
sim_n_control = 1200
sim_conv_control = 180
sim_n_treat = 1180
sim_conv_treat = 205
```


In [ ]:
# TODO: implement the simulation function that returns P(better) and expected losses
pass


## 9. Reflection – Audience Adaptation

Write three short bullet points describing how you would present “P(treatment better) ≈ 0.94” to:
- a C-level executive
- a data-science peer
- a mixed product / design audience


In [ ]:
print("Reflection notes go here.")
